# N-CSR 前缀合并页面零网络修复

同一已封存 SEC 页面包含 N-CSR、N-CSRS 及修订形式；只修解析，不重新请求。


In [1]:
import json,hashlib,re,collections
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_series_filing_search_20261023';plan=json.loads((card/'additional_forms_parse_repair_plan.json').read_text())
assert plan['frozen'] and plan['budget']['additional_HTTP_GETs']==0
assert json.loads((card/'additional_forms_result.json').read_text())['status']=='additional_series_index_partial_or_unqualified'
raw=root/plan['raw_page'];assert hashlib.sha256(raw.read_bytes()).hexdigest()==plan['raw_sha256']
soup=BeautifulSoup(raw.read_bytes(),'html.parser');identity=soup.select_one('p.identInfo');assert identity is not None and 'S000004354' in identity.get_text(' ',strip=True) and 'Semiconductor ETF' in identity.get_text(' ',strip=True)
table=soup.select_one('#seriesDiv table.tableFile2');assert table is not None
rows=[]
for tr in table.select('tr'):
 cells=tr.find_all('td',recursive=False)
 if not cells:continue
 assert len(cells)==5
 form=cells[0].get_text(' ',strip=True);assert form in {'N-CSR','N-CSRS','N-CSR/A','N-CSRS/A'}
 link=cells[1].find('a',id='documentsbutton');assert link and link.get('href')
 match=re.search(r'([0-9]{10}-[0-9]{2}-[0-9]{6})-index\.htm',link['href']);assert match
 filed=cells[3].get_text(' ',strip=True);assert re.fullmatch(r'20[0-9]{2}-[0-9]{2}-[0-9]{2}',filed)
 rows.append({'form':form,'accession':match.group(1),'filed':filed,'index_href':link['href']})
assert len(rows)==48 and len({x['accession'] for x in rows})==48
forms=dict(collections.Counter(x['form'] for x in rows));assert forms=={'N-CSR':22,'N-CSRS':21,'N-CSR/A':3,'N-CSRS/A':2}
primary=json.loads((root/'research/stage1_soxx_legacy_basket_history_20261023/candidate_trust_filings.json').read_text())['rows'];primary_ids={x['accessionNumber'] for x in primary}
base=[x for x in rows if '2011'<=x['filed'][:4]<='2018' and x['form'] in {'N-CSR','N-CSRS'}]
assert all(x['accession'] in primary_ids for x in base)
result={'status':'SEC_series_specific_N_CSR_N_CSRS_union_page_qualified_source_only','source_raw_sha256':plan['raw_sha256'],'source_NAS_archive_sha256':plan['nas_archive_sha256'],'source_NAS_scoped_restore_verified':True,'series_id':'S000004354','total_rows':len(rows),'form_counts':forms,'2011_2018_base_rows':len(base),'2011_2018_base_all_in_trust_inventory':True,'first_filing_date':min(x['filed'] for x in rows),'last_filing_date':max(x['filed'] for x in rows),'rows':rows,'new_HTTP_GETs':0,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_holdings_values_qualified':False}
(card/'additional_forms_parse_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','total_rows','form_counts','2011_2018_base_rows','first_filing_date','last_filing_date')})


{'status': 'SEC_series_specific_N_CSR_N_CSRS_union_page_qualified_source_only', 'total_rows': 48, 'form_counts': {'N-CSR': 22, 'N-CSRS': 21, 'N-CSRS/A': 2, 'N-CSR/A': 3}, '2011_2018_base_rows': 17, 'first_filing_date': '2006-04-10', 'last_filing_date': '2026-06-04'}


## 中文结论

从同一已备份 N-CSR 搜索页零网络读取 48 条系列索引：N-CSR 22、N-CSRS 21、N-CSR/A 3、N-CSRS/A 2。2011–2018 年基础表格 17 条 accession 均在信托目录；与 N-Q 合并后是 32 个不同报告期、5 条系列级修订候选。没有正文持仓值或交易时序资格。
